> **Development note:** This project includes explicit record counts, data previews, and reconciliation checks to make pipeline behavior observable during development and testing. In a production pipeline, unnecessary Spark actions such as repeated `count()` and `display()` calls would be removed or replaced with appropriate monitoring and data-quality metrics to reduce compute overhead.

# LA Parking Citations — Geospatial Data Pipeline

This project builds a medallion-style data pipeline in Databricks using Los Angeles parking citation data and City Council district boundaries.

The pipeline is organized into three notebooks and orchestrated as a Databricks Lakeflow Job:

1. **01_ingest_bronze** — Ingest parking citation data from the LA Open Data API and persist the source data to a Bronze Delta table.
2. **02_transform_silver** — Clean, standardize, validate, and quarantine citation records to produce a Silver Delta table.
3. **03_enrich_gold** — Enrich valid citations with City Council district boundaries and create an analytics-ready Gold Delta table.

### Pipeline Architecture

LA Open Data API  
↓  
**Bronze** — Raw parking citation data  
↓  
**Silver** — Cleaned, typed, and validated citations  
↓  
**Gold** — Geospatially enriched citation data

## Notebook 1: Bronze Ingestion

This notebook is responsible for the ingestion layer of the pipeline. It retrieves parking citation records from the LA Open Data API, applies an explicit source schema, performs basic source-data profiling, and persists the results to the Bronze Delta table.

**Output:** `parking_citations_bronze`

Downstream transformation and data-quality logic are handled by `02_transform_silver`.

In [0]:
import requests

# Fetch a very small sample first so we can inspect the source structure
# before scaling the ingestion up to thousands or millions of records.

API_LIMIT = 50000
OFFSET = 200000

url = "https://data.lacity.org/resource/4f5p-udkv.json"
params = {
        "$limit": API_LIMIT,
        "$offset": OFFSET,
           }

response = requests.get(url, params=params)
response.raise_for_status()

raw_data = response.json()


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

# Bronze layer: preserve the API response as close to the source as possible.
# Socrata returns many numeric-looking fields as strings, so all source fields
# are ingested as strings here and cast to appropriate types in Silver.
schema = StructType([
    StructField("ticket_number", StringType(), True),
    StructField("issue_date", StringType(), True),
    StructField("issue_time", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("marked_time", StringType(), True),
    StructField("rp_state_plate", StringType(), True),
    StructField("plate_expiry_date", StringType(), True),
    StructField("vin", StringType(), True),
    StructField("make", StringType(), True),
    StructField("body_style", StringType(), True),
    StructField("color", StringType(), True),
    StructField("location", StringType(), True),
    StructField("route", StringType(), True),
    StructField("agency", StringType(), True),
    StructField("violation_code", StringType(), True),
    StructField("violation_description", StringType(), True),
    StructField("fine_amount", StringType(), True),
    StructField("agency_desc", StringType(), True),
    StructField("color_desc", StringType(), True),
    StructField("body_style_desc", StringType(), True),
    StructField("loc_lat", StringType(), True),
    StructField("loc_long", StringType(), True),
    StructField("geocodelocation", StringType(), True)
])

In [0]:
# Bronze Dataframe
bronze_df = spark.createDataFrame(raw_data, schema=schema)

display(bronze_df.limit(10))

ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation
4603856084,2025-05-31T00:00:00.000,2113,null,0000,CA,202408,null,JEEP,PA,GY,850 7TH ST W,null,56,5200,DISPLAY OF PLATES,25,56 - DOT - CENTRAL,GREY,PASSENGER CAR,34.0390366,-118.2447298,"{'type': 'Point', 'coordinates': [-118.2447298, 34.0390366]}"
4603930730,2025-05-31T00:00:00.000,2134,null,0000,CA,202501,null,MERZ,PA,GY,5315 9TH ST W,null,51,80.58L,PREFERENTIAL PARKING,68,51 - DOT - WESTERN,GREY,PASSENGER CAR,34.05878999,-118.34604232,"{'type': 'Point', 'coordinates': [-118.34604232, 34.05878999]}"
4604402014,2025-05-31T00:00:00.000,1456,null,0000,CA,202302,null,PORS,PA,GY,146 LAUREL AV S,null,54,4000(a)1,EXPIRED REGISTRATION,50,54 - DOT - HOLLYWOOD,GREY,PASSENGER CAR,34.07277029,-118.36464923,"{'type': 'Point', 'coordinates': [-118.36464923, 34.07277029]}"
4604155253,2025-05-31T00:00:00.000,1659,SO14A,0000,VT,202407,null,JEEP,PA,BK,14501 VENTURA BL,null,53,88.13B+,METER EXP.,63,53 - DOT - VALLEY,BLACK,PASSENGER CAR,34.1513168,-118.450474,"{'type': 'Point', 'coordinates': [-118.450474, 34.1513168]}"
4603930601,2025-05-31T00:00:00.000,1823,null,0000,CA,202504,null,TOYT,PA,BK,744 DUNSMUIR AV,null,51,80.58L,PREFERENTIAL PARKING,68,51 - DOT - WESTERN,BLACK,PASSENGER CAR,34.06100693,-118.34874459,"{'type': 'Point', 'coordinates': [-118.34874459, 34.06100693]}"
4604347801,2025-05-31T00:00:00.000,1447,null,0000,CA,202601,null,MAZD,PA,BK,133 FLORES ST S,null,54,5200,DISPLAY OF PLATES,25,54 - DOT - HOLLYWOOD,BLACK,PASSENGER CAR,34.07349098,-118.37118875,"{'type': 'Point', 'coordinates': [-118.37118875, 34.07349098]}"
4603783671,2025-05-31T00:00:00.000,1455,null,0000,CA,202601,null,AUDI,PA,BK,665 MARIPOSA AV S,null,54,80.56E4+,RED ZONE,93,54 - DOT - HOLLYWOOD,BLACK,PASSENGER CAR,34.0614794,-118.2988131,"{'type': 'Point', 'coordinates': [-118.2988131, 34.0614794]}"
4604015021,2025-05-31T00:00:00.000,1915,null,0000,CA,202509,null,TOYT,PA,WT,4173 DIXIE CANYON AV,null,53,80.58L,PREFERENTIAL PARKING,68,53 - DOT - VALLEY,WHITE,PASSENGER CAR,34.1459983,-118.4249253,"{'type': 'Point', 'coordinates': [-118.4249253, 34.1459983]}"
4604433750,2025-05-31T00:00:00.000,1624,SS128,0000,CA,202510,null,BMW,PA,GY,11300 IDAHO AV,null,51,88.13B+,METER EXP.,63,51 - DOT - WESTERN,GREY,PASSENGER CAR,34.04570223,-118.44838483,"{'type': 'Point', 'coordinates': [-118.44838483, 34.04570223]}"
4604016244,2025-05-31T00:00:00.000,1858,null,0000,CA,202501,null,NISS,PA,OT,11300 GLENOAKS BLVD,null,53,80.61,STANDNG IN ALLEY,68,53 - DOT - VALLEY,OTHER,PASSENGER CAR,34.2751736,-118.41408033,"{'type': 'Point', 'coordinates': [-118.41408033, 34.2751736]}"


In [0]:
# Bronze Schema Check
bronze_df.printSchema()

root
 |-- ticket_number: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- issue_time: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- marked_time: string (nullable = true)
 |-- rp_state_plate: string (nullable = true)
 |-- plate_expiry_date: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- make: string (nullable = true)
 |-- body_style: string (nullable = true)
 |-- color: string (nullable = true)
 |-- location: string (nullable = true)
 |-- route: string (nullable = true)
 |-- agency: string (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- fine_amount: string (nullable = true)
 |-- agency_desc: string (nullable = true)
 |-- color_desc: string (nullable = true)
 |-- body_style_desc: string (nullable = true)
 |-- loc_lat: string (nullable = true)
 |-- loc_long: string (nullable = true)
 |-- geocodelocation: string (nullable = true)



### Source Data Quality Profile

Before persisting the Bronze layer, perform lightweight profiling of the source data. These checks identify missing values and potential source-data quality issues without modifying the records.

Cleaning and validation are deferred to the Silver layer.

In [0]:
# Bronze Profile
record_count = bronze_df.count()
print(f"Bronze records: {record_count:,}")

Bronze records: 50,000


In [0]:
from pyspark.sql import functions as F

display(
    bronze_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("ticket_number").alias("unique_tickets"),
        F.sum(F.col("ticket_number").isNull().cast("int")).alias("null_ticket_numbers")
    )
)

total_rows,unique_tickets,null_ticket_numbers
50000,50000,0


In [0]:
null_counts = bronze_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
])

display(null_counts)

ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation
0,0,0,40971,0,0,0,50000,296,2505,534,25,43761,0,1061,1061,0,0,2526,3009,0,0,25


## Persist Bronze Layer

Persist the ingested parking citation data as a Delta table for use by downstream pipeline tasks.

During development, the existing Bronze table is dropped and recreated to keep repeated runs reproducible. After writing, the table is read back to verify that the Bronze data was persisted successfully.

> **Development note:** A production pipeline would typically use an incremental or idempotent ingestion strategy rather than dropping and recreating the table on each run.

In [0]:
# Reset Bronze Table
spark.sql("DROP TABLE IF EXISTS parking_citations_bronze")

DataFrame[]

In [0]:
# Persist Bronze Delta Table
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("parking_citations_bronze")
)

In [0]:
# Verify Table
bronze_table = spark.table("parking_citations_bronze")

print(f"Persisted Bronze records: {bronze_table.count():,}")
display(bronze_table.limit(10))

Persisted Bronze records: 50,000


ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation
4603959006,2025-06-05T00:00:00.000,902,null,0000,CA,202409,null,DODG,VN,BK,20700 BRYANT ST,null,53,80.69BS,NO PARK/STREET CLEAN,73,53 - DOT - VALLEY,BLACK,VAN,34.2269981,-118.5861034,"{'type': 'Point', 'coordinates': [-118.5861034, 34.2269981]}"
4604675740,2025-06-05T00:00:00.000,837,null,0000,CA,202601,null,AUDI,PA,GY,3118 LA SALLE AVE S,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,GREY,PASSENGER CAR,34.02576288,-118.30555308,"{'type': 'Point', 'coordinates': [-118.30555308, 34.02576288]}"
4604443071,2025-06-05T00:00:00.000,845,null,0000,CA,202512,null,HYUN,PA,WT,3795 ROSEWOOD AV,null,51,80.69BS,NO PARK/STREET CLEAN,73,51 - DOT - WESTERN,WHITE,PASSENGER CAR,33.99900646,-118.44362349,"{'type': 'Point', 'coordinates': [-118.44362349, 33.99900646]}"
4604637715,2025-06-05T00:00:00.000,1233,null,0000,CA,202501,null,HYUN,PA,BK,1330 35TH PL W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,BLACK,PASSENGER CAR,34.02371719,-118.29833515,"{'type': 'Point', 'coordinates': [-118.29833515, 34.02371719]}"
4604657960,2025-06-05T00:00:00.000,1040,null,0000,AZ,202509,null,TOYT,PA,BK,5216 ROSELAND ST,null,55,80.58L,PREFERENTIAL PARKING,68,55 - DOT - SOUTHERN,BLACK,PASSENGER CAR,34.02841185,-118.35784251,"{'type': 'Point', 'coordinates': [-118.35784251, 34.02841185]}"
4604096151,2025-06-05T00:00:00.000,856,HO667,0000,CA,202506,null,TOYT,PA,BL,1701 CHEROKEE AV,null,54,88.13B+,METER EXP.,63,54 - DOT - HOLLYWOOD,BLUE,PASSENGER CAR,34.1019943,-118.3347587,"{'type': 'Point', 'coordinates': [-118.3347587, 34.1019943]}"
4604217914,2025-06-03T00:00:00.000,1016,null,0000,CA,202510,null,TOYT,PA,WT,3120 71ST ST W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,WHITE,PASSENGER CAR,33.9754104,-118.3273543,"{'type': 'Point', 'coordinates': [-118.3273543, 33.9754104]}"
4604735450,2025-06-05T00:00:00.000,1225,null,0000,CA,202405,null,HOND,PA,GY,1144 39TH ST W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,GREY,PASSENGER CAR,34.0155554,-118.2956359,"{'type': 'Point', 'coordinates': [-118.2956359, 34.0155554]}"
4604527793,2025-06-05T00:00:00.000,856,null,0000,CA,202606,null,HOND,PA,SL,5000 8TH ST,null,54,80.69BS,NO PARK/STREET CLEAN,73,54 - DOT - HOLLYWOOD,SILVER,PASSENGER CAR,34.06051353,-118.33849479,"{'type': 'Point', 'coordinates': [-118.33849479, 34.06051353]}"
4604471966,2025-06-05T00:00:00.000,830,null,0000,CA,0,null,CHEV,PA,BK,1771 35TH ST W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,BLACK,PASSENGER CAR,34.02478517,-118.31095343,"{'type': 'Point', 'coordinates': [-118.31095343, 34.02478517]}"
